# Taller de Regresión Lineal y KNN

## Parte 1: Regresión Lineal (Precios de Viviendas)
Primero, importamos todas las librerías necesarias para el análisis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import time

### 1.1 Carga del Dataset (Boston Housing)
Cargamos el dataset `HousingData.csv`. Este dataset contiene información sobre viviendas en Boston, y la variable objetivo a predecir es `MEDV` (Precio mediano en miles de dólares). Como tiene algunos valores nulos, los rellenaremos con la media de sus columnas.

In [ ]:
# Cargar el dataset real
df_housing = pd.read_csv('HousingData.csv')

# Tratar valores nulos (NaN) usando la media de cada columna
df_housing = df_housing.fillna(df_housing.mean())


### 1.2 Exploración Básica
Visualizamos las primeras filas y las estadísticas descriptivas.

In [ ]:
display(df_housing.head())
display(df_housing.describe())

### 1.3 Análisis Exploratorio de Datos (EDA)
Graficamos la relación entre algunas variables clave (ej. RM: Habitaciones, AGE: Edad, LSTAT: % Estatus Bajo) y el precio (MEDV).

In [ ]:
plt.figure(figsize=(15, 4))
variables_eda = ['RM', 'AGE', 'LSTAT']
for i, col in enumerate(variables_eda):
    plt.subplot(1, 3, i+1)
    sns.scatterplot(x=df_housing[col], y=df_housing['MEDV'], alpha=0.6)
    plt.title(f'MEDV (Precio) vs {col}')
plt.tight_layout()
plt.show()

### Preparación de los datos
Separamos las variables características (X) de nuestra variable objetivo (y). Usaremos todas las variables excepto MEDV. Además, normalizamos los datos, lo cual es vital para que el Gradiente Descendente converja.

In [ ]:
X = df_housing.drop(columns=['MEDV']).values
y = df_housing['MEDV'].values.reshape(-1, 1)

# Normalización para Gradiente Descendente
X_mean = np.mean(X, axis=0)
X_std = np.std(X, axis=0)
X_norm = (X - X_mean) / X_std

# Añadimos la columna de unos (x0) para el término del intercepto
X_full_norm = np.c_[np.ones((len(X_norm), 1)), X_norm]
X_full = np.c_[np.ones((len(X), 1)), X]

## Parte 2: Regresión Lineal con Gradiente Descendente
Implementamos las funciones desde cero e iteramos calculando el costo y el gradiente.

In [ ]:
def costo_mse(X, y, theta):
    m = len(y)
    predicciones = X.dot(theta)
    return (1 / (2 * m)) * np.sum(np.square(predicciones - y))

def gradiente_descendente(X, y, theta, learning_rate, iteraciones):
    m = len(y)
    historial_costo = np.zeros(iteraciones)
    
    for i in range(iteraciones):
        predicciones = X.dot(theta)
        errores = predicciones - y
        gradiente = (1 / m) * X.T.dot(errores)
        theta = theta - learning_rate * gradiente
        historial_costo[i] = costo_mse(X, y, theta)
        
    return theta, historial_costo

theta_inicial = np.zeros((X_full_norm.shape[1], 1))
iteraciones = 1000
alpha = 0.1

start_time = time.time()
theta_gd, historial_costo = gradiente_descendente(X_full_norm, y, theta_inicial, alpha, iteraciones)
tiempo_gd = time.time() - start_time

pred_gd = X_full_norm.dot(theta_gd)
mse_gd = mean_squared_error(y, pred_gd)
print(f"MSE Gradiente Descendente: {mse_gd:.2f}")

### Evolución del error (Gradiente Descendente)
A continuación verificamos visualmente cómo el error cae y converge a medida que avanzan las iteraciones.

In [ ]:
plt.plot(range(iteraciones), historial_costo)
plt.title("Evolución del Error (MSE) vs Iteraciones")
plt.xlabel("Iteraciones")
plt.ylabel("Costo (Errores Cuadráticos)")
plt.show()

## Parte 3: Solución Analítica con Ecuación Normal
Aplicamos directamente el cálculo de theta utilizando álgebra matricial. $\theta = (X^T X)^{-1} X^T y$

In [ ]:
start_time = time.time()
# Obtenemos los parámetros exactos
theta_normal = np.linalg.inv(X_full.T.dot(X_full)).dot(X_full.T).dot(y)
tiempo_normal = time.time() - start_time

pred_normal = X_full.dot(theta_normal)
mse_normal = mean_squared_error(y, pred_normal)
print(f"MSE Ecuación Normal: {mse_normal:.2f}")

## Parte 4: Uso de librerías (Sklearn)
Realizamos el mismo proceso usando la clase altamente optimizada `LinearRegression` de scikit-learn.

In [ ]:
start_time = time.time()
modelo_sk = LinearRegression()
modelo_sk.fit(X, y)
tiempo_sk = time.time() - start_time

pred_sk = modelo_sk.predict(X)
mse_sk = mean_squared_error(y, pred_sk)
r2_sk = r2_score(y, pred_sk)

print(f"MSE Sklearn: {mse_sk:.2f}")
print(f"R2 Score: {r2_sk:.4f}")

## Parte 5: Comparación y Resultados
Tabla comparativa mostrando la eficiencia en predicción, velocidad y complejidad computacional.

In [ ]:
print("--- Tabla Comparativa ---")
print(f"{'Método':<25} | {'MSE':<15} | {'Tiempo (s)':<10} | {'Complejidad'}")
print("-" * 75)
print(f"{'Gradiente Descendente':<25} | {mse_gd:<15.2f} | {tiempo_gd:<10.6f} | O(k * n * d)")
print(f"{'Ecuación Normal':<25} | {mse_normal:<15.2f} | {tiempo_normal:<10.6f} | O(d^3)")
print(f"{'sklearn':<25} | {mse_sk:<15.2f} | {tiempo_sk:<10.6f} | Optimizada")

## Parte 6: Interpretación del Modelo
¿Qué nos dicen los coeficientes obtenidos? Mostramos algunos ejemplos de las features del Boston Housing Dataset:

In [ ]:
print("--- Interpretación de Coeficientes (sklearn) ---")
print(f"Intercepto base: {modelo_sk.intercept_[0]:.2f}")
features = df_housing.drop(columns=['MEDV']).columns
for i, col in enumerate(features[:5]): # Mostramos los 5 primeros coeficientes para no saturar
    print(f"Coeficiente {col}: {modelo_sk.coef_[0][i]:.2f}")
print("...")

# Práctica: K-Nearest Neighbors (KNN)

## Parte A: Implementación Base y Carga de Dataset Iris
Cargamos la información de la librería y separamos en 70% Train y 30% Test.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from collections import Counter

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

X_train, X_test, y_train, y_test = train_test_split(X_iris, y_iris, test_size=0.3, random_state=42)
print("Datos cargados exitosamente.")

### Funciones Base (KNN desde Cero)
Funciones de distancia euclidiana/manhattan y predicción para voto por mayoría.

In [ ]:
def calcular_distancia(x1, x2, tipo='euclidiana'):
    if tipo == 'euclidiana':
        return np.sqrt(np.sum((x1 - x2)**2))
    elif tipo == 'manhattan':
        return np.sum(np.abs(x1 - x2))

def predecir_knn(X_train, y_train, x_test, k, tipo_distancia='euclidiana'):
    distancias = [calcular_distancia(x_test, x_train, tipo_distancia) for x_train in X_train]
    indices_k_cercanos = np.argsort(distancias)[:k]
    etiquetas_k_cercanos = [y_train[i] for i in indices_k_cercanos]
    voto_mayoria = Counter(etiquetas_k_cercanos).most_common(1)[0][0]
    return voto_mayoria

def knn_exactitud(X_train, y_train, X_test, y_test, k, tipo_distancia='euclidiana'):
    predicciones = [predecir_knn(X_train, y_train, x, k, tipo_distancia) for x in X_test]
    return np.mean(predicciones == y_test)

## Parte B: Experimentos
### Experimento 1: Variar K
Miramos el efecto del K en entrenamiento vs pruebas para detectar sobreajuste (overfitting).

In [ ]:
print("--- Experimento 1: Variar K ---")
for k in [1, 3, 5, 7, 9]:
    acc_test = knn_exactitud(X_train, y_train, X_test, y_test, k)
    acc_train = knn_exactitud(X_train, y_train, X_train, y_train, k)
    print(f"K={k}: Accuracy en Test = {acc_test:.4f} | Accuracy en Train = {acc_train:.4f}")

# Nota: Cuando K=1 el Train es 1.0 (100%), ahí ocurre Overfitting absoluto.

### Experimento 2: Normalización
Aplicamos Min-Max scaler y Z-score scaler desde cero y lo probamos.

In [ ]:
# 1. Min-Max Normalización
min_val = X_train.min(axis=0)
max_val = X_train.max(axis=0)
X_train_minmax = (X_train - min_val) / (max_val - min_val)
X_test_minmax = (X_test - min_val) / (max_val - min_val)

# 2. Z-score Normalización
mean_val = X_train.mean(axis=0)
std_val = X_train.std(axis=0)
X_train_z = (X_train - mean_val) / std_val
X_test_z = (X_test - mean_val) / std_val

k_optimo = 3
acc_orig = knn_exactitud(X_train, y_train, X_test, y_test, k_optimo)
acc_minmax = knn_exactitud(X_train_minmax, y_train, X_test_minmax, y_test, k_optimo)
acc_z = knn_exactitud(X_train_z, y_train, X_test_z, y_test, k_optimo)

print("--- Comparativa Normalizaciones (K=3) ---")
print(f"Sin normalizar : {acc_orig:.4f}")
print(f"Min-Max        : {acc_minmax:.4f}")
print(f"Z-score        : {acc_z:.4f}")

### Experimento 3: Distancias
Evaluamos el impacto de utilizar distancia Euclidiana vs Manhattan.

In [ ]:
print("--- Experimento 3: Tipos de Distancia ---")
acc_euclidiana = knn_exactitud(X_train, y_train, X_test, y_test, k_optimo, 'euclidiana')
acc_manhattan = knn_exactitud(X_train, y_train, X_test, y_test, k_optimo, 'manhattan')

print(f"Euclidiana Acc: {acc_euclidiana:.4f}")
print(f"Manhattan Acc:  {acc_manhattan:.4f}")